In [7]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "src").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not locate project root containing src/")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

SRC = PROJECT_ROOT / "src"
ARTIFACTS = PROJECT_ROOT / "artifacts"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CWD:", Path.cwd())
print("ARTIFACTS:", ARTIFACTS)

PROJECT_ROOT: .
CWD: .
ARTIFACTS: ./artifacts


In [8]:
from ips_learning.visualization.utils import StudyFigureRepository
figure_repo = StudyFigureRepository(root_dir=ARTIFACTS)

In [9]:
SYSTEM_NAMES = ["spp", "levi", "kuramoto"]
PROFILE_NAMEs = ["spp_velocity_grid_clumps", "levi_velocity_grid", "profile1"]

SYSTEM_ORDERs = [2, 2, 1]
INTERACTION_TYPE = ["energy", "alignment", "energy"]
ENVIRONMENT_VARIABLE = ["velocity", "velocity", "position"]

D = 2


In [10]:
FIGURE_STAGE = "step_6C2"

df = {}

for SYSTEM_NAME in SYSTEM_NAMES:

    ml_eval_dir = (
        ARTIFACTS
        / SYSTEM_NAME
        / "evaluations"
        / "ML_sweep"
    )
    trial_metrics_path = (
        ml_eval_dir
        / "ML_sweep_trial_metrics.csv"
    )

    summary_metrics_path = (
        ml_eval_dir
        / "ML_sweep_metrics_mean_std.csv"
    )

    # One row per (M, L), containing statistics across trials.
    df_ml = pd.read_csv(summary_metrics_path)

    df[SYSTEM_NAME] = df_ml


In [19]:
# --------------------------------------------------
# Helpers for combined multi-system heatmaps
# --------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import Normalize

from ips_learning.visualization.utils import StudyFigureRepository
from ips_learning.visualization.style import apply_paper_style

apply_paper_style()
figure_repo = StudyFigureRepository(root_dir=ARTIFACTS)


SYSTEM_DISPLAY_NAMES = {
    "spp": "SPP",
    "levi": "Phototaxis",
    "kuramoto": "Kuramoto",
}

def pivot_metric(dataframe, metric):
    """
    Convert one system's long-form M/L table into a heatmap matrix.

    Rows correspond to M and columns correspond to L.
    """
    return (
        dataframe
        .pivot(
            index="M",
            columns="L",
            values=metric,
        )
        .sort_index(axis=0)
        .sort_index(axis=1)
    )


def prepare_heatmap_values(
    dataframe,
    metric,
    *,
    log_scale=True,
):
    """
    Return the pivoted DataFrame and the values to plot.
    """
    pivot_df = pivot_metric(
        dataframe,
        metric,
    )

    values = pivot_df.to_numpy(dtype=float)

    if log_scale:
        values = np.where(
            values > 0,
            np.log10(values),
            np.nan,
        )

    return pivot_df, values


def plot_all_system_heatmaps(
    df_by_system,
    *,
    metric,
    cbar_label,
    system_order=("spp", "levi", "kuramoto"),
    log_scale=True,
):
    """
    Plot one metric for all systems in a single row.

    All panels use the same normalization and one shared colorbar.
    """
    prepared = {}

    # --------------------------------------------------
    # Prepare every system before plotting
    # --------------------------------------------------
    for system_name in system_order:
        if system_name not in df_by_system:
            raise KeyError(
                f"Missing DataFrame for system {system_name!r}."
            )

        system_df = df_by_system[system_name]

        if metric not in system_df.columns:
            raise KeyError(
                f"Column {metric!r} is missing from "
                f"the {system_name!r} summary CSV."
            )

        pivot_df, values = prepare_heatmap_values(
            system_df,
            metric,
            log_scale=log_scale,
        )

        prepared[system_name] = {
            "pivot": pivot_df,
            "values": values,
        }

    # --------------------------------------------------
    # Compute one color range over all systems
    # --------------------------------------------------
    finite_values = []

    for system_name in system_order:
        values = prepared[system_name]["values"]

        finite_values.append(
            values[np.isfinite(values)]
        )

    finite_values = [
        values
        for values in finite_values
        if values.size > 0
    ]

    if not finite_values:
        raise ValueError(
            f"No finite values are available for {metric!r}."
        )

    combined_values = np.concatenate(
        finite_values
    )

    shared_vmin = float(
        np.nanmin(combined_values)
    )

    shared_vmax = float(
        np.nanmax(combined_values)
    )

    # Avoid an invalid normalization when all values are equal.
    if np.isclose(shared_vmin, shared_vmax):
        shared_vmin -= 0.5
        shared_vmax += 0.5

    shared_norm = Normalize(
        vmin=shared_vmin,
        vmax=shared_vmax,
    )

    # --------------------------------------------------
    # Construct the combined figure
    # --------------------------------------------------
    n_systems = len(system_order)

    fig, axes = plt.subplots(
        1,
        n_systems,
        figsize=(5.0 * n_systems, 5.2),
        constrained_layout=True,
    )

    axes = np.atleast_1d(axes)

    image = None

    for ax, system_name in zip(
        axes,
        system_order,
    ):
        pivot_df = prepared[system_name]["pivot"]
        values = prepared[system_name]["values"]

        image = ax.imshow(
            values,
            origin="lower",
            aspect="auto",
            norm=shared_norm,
        )

        ax.set_xticks(
            np.arange(len(pivot_df.columns))
        )
        ax.set_xticklabels(
            pivot_df.columns
        )

        ax.set_yticks(
            np.arange(len(pivot_df.index))
        )
        ax.set_yticklabels(
            pivot_df.index
        )

        ax.set_xlabel(r"$L$")
        ax.set_ylabel(r"$M$")

        ax.set_title(
            SYSTEM_DISPLAY_NAMES.get(
                system_name,
                system_name,
            )
        )
        ax.grid(False)

    # --------------------------------------------------
    # Add one colorbar shared by every panel
    # --------------------------------------------------
    colorbar = fig.colorbar(
        image,
        ax=axes.tolist(),
        location="right",
        shrink=0.88,
        pad=0.02,
    )

    if log_scale:
        colorbar.set_label(
            rf"$\log_{{10}}\left({cbar_label}\right)$"
        )
    else:
        colorbar.set_label(cbar_label)

    return fig

In [ ]:
# --------------------------------------------------
# Combined heatmaps across all systems
# One figure per learned feature
# --------------------------------------------------
mean_metric_specs = [
    {
        "column": "E_phi_relative_mean",
        "label": r"\overline{E}_{\phi}^{\mathrm{rel}}",
        "file_name": (
            "ML_sweep_all_systems_"
            "E_phi_relative_mean_heatmaps"
        ),
    },
    {
        "column": "E_env_relative_mean",
        "label": r"\overline{E}_{f}^{\mathrm{rel}}",
        "file_name": (
            "ML_sweep_all_systems_"
            "E_env_relative_mean_heatmaps"
        ),
    },
]


for spec in mean_metric_specs:
    fig = plot_all_system_heatmaps(
        df,
        metric=spec["column"],
        cbar_label=spec["label"],
        system_order=(
            "spp",
            "levi",
            "kuramoto",
        ),
        log_scale=True,
    )

    path = figure_repo.save(
        fig,

        # These are combined-system figures, so do not save
        # them under one individual physical system.
        study_name="training_data_dependence",

        stage_name=FIGURE_STAGE,
        file_name=spec["file_name"],
        close=False,
    )

    print("Saved:", path)

    plt.show()